## Resources

In [13]:
pip install flashrank


In [14]:
pip install sentence-transformers


In [15]:
pip install rankify

In [16]:
pip install chromadb

In [17]:
pip install rank-bm25

Create Data for testing

In [18]:
import pandas as pd
import json

# Simulation of Raw S3 Object Storage Data Records
raw_s3_bucket = [
    {
        "storage_uri": "s3://central-bank-vault/policies/risk_matrix_2026.txt",
        "title": "2026 Internal Risk Mitigation and Infrastructure Architecture Protocols",
        "acl_permitted_roles": ["Compliance_Officer", "Audit_Team", "System_Admin"],
        "content": (
            "This document dictates infrastructure safety criteria. When processing capital transfers, "
            "systems must cross-reference core ledger systems. Below is the active asset classification matrix:\n\n"
            "| Component ID | Subsystem Target | Risk Vulnerability Level |\n"
            "| :--- | :--- | :--- |\n"
            "| AUTH-881 | User Credentials Layer | Medium |\n"
            "| RISK-992-ALPHA | High-Frequency Transaction Pipeline | CRITICAL |\n"
            "| STOR-004 | In-Memory Vault Mirroring | High |\n\n"
            "Warning: Any structural unexpected downtime on asset RISK-992-ALPHA requires immediate rolling "
            "hot-swaps to secondary nodes to prevent international reconciliation clearing failures."
        )
    },
    {
        "storage_uri": "s3://central-bank-vault/hr/executive_compensation_q1.txt",
        "title": "Q1 Executive Leadership Compensations and Active Payroll Tier Schema",
        "acl_permitted_roles": ["Executive_Board", "HR_Director"],
        "content": (
            "Private Document - Strictly Restricted. Executive Salary distribution schedules for the fiscal year 2026. "
            "Chief Executive Officer base compensation is set at $450,000 per quarter. Chief Technology Officer "
            "base tier calculation stands at $380,000 per quarter. Discretionary performance bonuses are bound "
            "by the asset allocation thresholds specified under HR-COMP-2026."
        )
    }
]

print(f"Ingested {len(raw_s3_bucket)} raw objects from simulated S3 storage.")

Ingested 2 raw objects from simulated S3 storage.


Parent Child chunk

In [19]:
import re

parent_document_store = {}
all_child_chunks = []

for doc in raw_s3_bucket:
    # Use storage_uri as the primary unique key across our distributed data landscape
    parent_id = doc["storage_uri"]

    # Store the pristine, complete raw string as the Parent Context
    parent_document_store[parent_id] = {
        "text": doc["content"],
        "title": doc["title"],
        "acl": doc["acl_permitted_roles"]
    }

    # Parser Layer: Extract child rows/sentences while strictly maintaining structure
    raw_lines = re.split(r'(?<=[.!?])\s+|\n', doc["content"])

    for index, line in enumerate(raw_lines):
        clean_line = line.strip()
        if len(clean_line) < 15: # Filter out raw layout artifact noise
            continue

        all_child_chunks.append({
            "child_id": f"{parent_id}#chunk_{index}",
            "parent_id": parent_id,
            "text": clean_line,
            "metadata": {
                "source_title": doc["title"],
                "permitted_roles": json.dumps(doc["acl_permitted_roles"]) # Meta filters require primitive serialization
            }
        })

print(f"Constructed {len(all_child_chunks)} atomic child nodes linked back to parents.")

Constructed 14 atomic child nodes linked back to parents.


The vecotr Db input( multi Engine indexing)

In [20]:
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

# 1. Hydrate Dense Vector Database Engine (Simulating Clustered In-Memory Configuration)
chroma_client = chromadb.Client()
# Employs a local SentenceTransformer model running completely on CPU
embedding_engine = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
#vector_db = chroma_client.create_collection(name="secure_banking_vault", embedding_function=embedding_engine)
vector_db = chroma_client.get_or_create_collection(
    name="secure_banking_vault",
    embedding_function=embedding_engine
)

# Prepare vectors packaging data arrays
c_ids = [item["child_id"] for item in all_child_chunks]
c_texts = [item["text"] for item in all_child_chunks]
c_metadatas = [item["metadata"] for item in all_child_chunks]

vector_db.add(ids=c_ids, documents=c_texts, metadatas=c_metadatas)

# 2. Hydrate Sparse Lexical Database Engine (BM25 Framework)
tokenized_clean_corpus = [text.lower().split(" ") for text in c_texts]
bm25_engine = BM25Okapi(tokenized_clean_corpus)

print("Dual-Indexing Layer complete. Dense vector graphs and Sparse token weight tables are live.")


Dual-Indexing Layer complete. Dense vector graphs and Sparse token weight tables are live.


Hybrid retrieval secure

In [21]:

def secure_hybrid_retrieval(query, user_active_roles, top_k=3):
    query_tokens = query.lower().split(" ")
    fetch_pool_limit = max(top_k * 3, 10)
    # --- PHASE 1: DENSE VECTOR SEARCH WITH METADATA ACL HARD-FILTERING ---
    # Database-level filter: Iterate and evaluate if user roles overlap with document rules
    dense_results = vector_db.query(
        query_texts=[query],
        n_results=fetch_pool_limit # Pull a wider candidate pool for RRF to evaluate
    )

    filtered_dense_ids = []
    if dense_results["ids"] and dense_results["ids"][0]:
        for idx, cid in enumerate(dense_results["ids"][0]):
            meta = dense_results["metadatas"][0][idx]
            permitted = json.loads(meta["permitted_roles"])

            # Security Rule Check: If intersection exists, user is authenticated for this chunk
            if any(role in user_active_roles for role in permitted):
                filtered_dense_ids.append(cid)

    # --- PHASE 2: SPARSE KEYWORD SEARCH ---
    bm25_scores = bm25_engine.get_scores(query_tokens)
    sorted_sparse_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)

    filtered_sparse_ids = []
    for idx in sorted_sparse_indices:
        child_item = all_child_chunks[idx]
        permitted = json.loads(child_item["metadata"]["permitted_roles"])
        if any(role in user_active_roles for role in permitted):
            filtered_sparse_ids.append(child_item["child_id"])
            if len(filtered_sparse_ids) >= 10:
                break

    # --- PHASE 3: RECIPROCAL RANK FUSION (RRF) ---
    rrf_registry = {}
    smoothing_constant = 60

    for rank, cid in enumerate(filtered_dense_ids):
        rrf_registry[cid] = rrf_registry.get(cid, 0.0) + (1.0 / (smoothing_constant + (rank + 1)))

    for rank, cid in enumerate(filtered_sparse_ids):
        rrf_registry[cid] = rrf_registry.get(cid, 0.0) + (1.0 / (smoothing_constant + (rank + 1)))

    fused_rankings = sorted(rrf_registry.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in fused_rankings[:top_k]]

print("Secure Hybrid Retrieval engine initialized.")

Secure Hybrid Retrieval engine initialized.


Cross encoder

In [22]:
from flashrank import Ranker, RerankRequest


# Initialize ultra-light cross-encoder model locally
reranking_model = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/tmp/")

auditor_query = "What happens if asset RISK-992-ALPHA goes down?"
auditor_session_roles = ["Audit_Team"]

# Step 1: Secure Hybrid Retrieval Pulls Candidate Material
fused_candidates = secure_hybrid_retrieval(auditor_query, auditor_session_roles, top_k=5)

# Step 2: Map IDs back to their text strings to build the Reranker payload
passages_to_score = []
for cid in fused_candidates:
    matching_node = next(node for node in all_child_chunks if node["child_id"] == cid)
    passages_to_score.append({
        "id": matching_node["child_id"],
        "text": matching_node["text"],
        "parent_id": matching_node["parent_id"] # Retain the parent tracker map
    })

# Step 3: Run full Cross-Attention Scoring
rerank_execution = RerankRequest(query=auditor_query, passages=passages_to_score)
reranked_output = reranking_model.rerank(rerank_execution)

print("\n--- POST-RERANKING AUDIT SCORES ---")
for position, item in enumerate(reranked_output):
    print(f"Rank {position+1} [ID: {item['id']}]: Score={item['score']:.4f} | Text: {item['text'][:80]}...")

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 59.8MiB/s]



--- POST-RERANKING AUDIT SCORES ---
Rank 1 [ID: s3://central-bank-vault/policies/risk_matrix_2026.txt#chunk_10]: Score=0.9989 | Text: Warning: Any structural unexpected downtime on asset RISK-992-ALPHA requires imm...
Rank 2 [ID: s3://central-bank-vault/policies/risk_matrix_2026.txt#chunk_7]: Score=0.5782 | Text: | RISK-992-ALPHA | High-Frequency Transaction Pipeline | CRITICAL |...
Rank 3 [ID: s3://central-bank-vault/policies/risk_matrix_2026.txt#chunk_2]: Score=0.0000 | Text: Below is the active asset classification matrix:...
Rank 4 [ID: s3://central-bank-vault/policies/risk_matrix_2026.txt#chunk_4]: Score=0.0000 | Text: | Component ID | Subsystem Target | Risk Vulnerability Level |...
Rank 5 [ID: s3://central-bank-vault/policies/risk_matrix_2026.txt#chunk_0]: Score=0.0000 | Text: This document dictates infrastructure safety criteria....


Context upscaling

In [23]:
# Extract the absolute winner from the reranked pipeline
winning_match = reranked_output[0]
target_parent_id = winning_match["parent_id"]

# Context Upscaling: Fetch the complete parent text block
upscaled_llm_context = parent_document_store[target_parent_id]["text"]
source_document_title = parent_document_store[target_parent_id]["title"]

# --- PRODUCTION GROUNDING GENERATION PROMPT ---
# Lock temperature to 0.0 at the API execution layer to force absolute greedy token selection.
deterministic_system_prompt = f"""
You are an unyielding, deterministic Bank Compliance Verification AI Assistant.
Your core task is to answer the User Query using ONLY the factual statements contained in the Verified Context provided below.

Rules:
1. If the answer cannot be explicitly derived from the Verified Context, say "ERROR: Information Not Found".
2. Do not use outside knowledge or extrapolate.
3. Provide the exact source document title as a citation.

Verified Context:
\"\"\"
{upscaled_llm_context}
\"\"\"

User Query: {auditor_query}
Answer:"""

print("\n--- FINAL PRODUCTION CONTEXT INJECTED PROMPT ---")
print(deterministic_system_prompt)


--- FINAL PRODUCTION CONTEXT INJECTED PROMPT ---

You are an unyielding, deterministic Bank Compliance Verification AI Assistant.
Your core task is to answer the User Query using ONLY the factual statements contained in the Verified Context provided below.

Rules:
1. If the answer cannot be explicitly derived from the Verified Context, say "ERROR: Information Not Found".
2. Do not use outside knowledge or extrapolate.
3. Provide the exact source document title as a citation.

Verified Context:
"""
This document dictates infrastructure safety criteria. When processing capital transfers, systems must cross-reference core ledger systems. Below is the active asset classification matrix:

| Component ID | Subsystem Target | Risk Vulnerability Level |
| :--- | :--- | :--- |
| AUTH-881 | User Credentials Layer | Medium |
| RISK-992-ALPHA | High-Frequency Transaction Pipeline | CRITICAL |
| STOR-004 | In-Memory Vault Mirroring | High |

"""

User Query: What happens if asset RISK-992-ALPHA go

his new layer directly is used to generate answer

In [28]:
pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00


In [29]:
from transformers import pipeline
import torch

print("\n--- LOADING LOCAL HUGGING FACE MODEL (ZERO KEYS REQUIRED) ---")

# 1. Initialize a text-generation pipeline using a lightweight, smart model
# We use device_map="auto" to automatically load it onto Colab's free GPU
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\n--- GENERATING COMPLIANCE ANSWER ---")

# 2. Pass your deterministic prompt straight into the model
outputs = generator(
    deterministic_system_prompt,
    max_new_tokens=150,
    temperature=0.0,    # Low temperature to keep it strict and factual
    do_sample=False     # Greedy decoding (equivalent to temperature 0.0)
)

# 3. Print out the text result
raw_text = outputs[0]["generated_text"]

# Clean up the output to print only the newly generated answer
final_answer = raw_text.split("Answer:")[-1].strip()

print("\n--- ACTUAL LOCAL COMPLIANCE ANSWER ---")
print(final_answer)


--- LOADING LOCAL HUGGING FACE MODEL (ZERO KEYS REQUIRED) ---


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


--- GENERATING COMPLIANCE ANSWER ---


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train 


--- ACTUAL LOCAL COMPLIANCE ANSWER ---
ERROR: Information Not Found

The given context does not provide information about what happens when asset RISK-992-ALPHA goes down. Therefore, I am unable to derive the required information and have returned an error message. The user query has been answered with the appropriate response based on the available data.


another way using gemini api

In [30]:
pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.27.2
    Uninstalling httpx-0.27.2:
      Successfully uninstalled httpx-0.27.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rankify 0.1.4 requires httpx==0.27.2, but you have httpx 0.28.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.43.0 which is incompatible.
google-adk 1.29.0 requires requests<3.0.0,>=2.32.4, but you have requests 2.32.3 which is incompatible.


In [33]:
import os
from google import genai
from google.genai import types

# 1. Attempt to safely pull from Colab's built-in Secrets Vault (The Key Icon on the left)
try:
    from google.colab import userdata
    if userdata.get('GEMINI_API_KEY'):
        os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

# 2. Guardrail Initialization Check
if os.environ.get("GEMINI_API_KEY"):
    client = genai.Client()
    print("🚀 Gemini Client successfully initialized using secure environment variables!")
else:
    print("⚠️ GEMINI_API_KEY not found in environment variables.")
    print("👉 Click the 'Key' icon on the left sidebar in Colab, add a secret named 'GEMINI_API_KEY', paste your key, and enable 'Notebook access'.")

🚀 Gemini Client successfully initialized using secure environment variables!


In [34]:
print("\n--- INITIATING GEMINI PRODUCTION GENERATION LAYER ---")

try:
    # We use 'gemini-2.5-flash' for ultra-fast, cost-effective compliance validation
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=deterministic_system_prompt,
        config=types.GenerateContentConfig(
            temperature=0.0, # Complete greedy token selection to force absolute rule adherence
        ),
    )

    print("\n--- ACTUAL PRODUCTION COMPLIANCE ANSWER ---")
    print(response.text)

except Exception as e:
    print(f"Execution failed. Ensure your API key is valid. Error: {e}")


--- INITIATING GEMINI PRODUCTION GENERATION LAYER ---

--- ACTUAL PRODUCTION COMPLIANCE ANSWER ---
Any structural unexpected downtime on asset RISK-992-ALPHA requires immediate rolling hot-swaps to secondary nodes to prevent international reconciliation clearing failures.
Citation: This document dictates infrastructure safety criteria.
